# 01. OpenAI API 기초
**SK하이닉스 Autonomous R&D — Day 3** 

> ChatGPT 웹과 **같은 모델**을 코드에서 호출.

---
## 0. 라이브러리 설치

아래 셀을 **최초 1회** 실행.

In [3]:
#노트북 환경에서 설치하거나, 터미널 환경에서 설치하거나
#노트북 a환경에서 설치하든, 터미널에서 a환경에서 설치하든 그건 관계없이 모두 적용된다.
#하지만 노트북 환경과 터미널 환경은 다르게 설정할 수 있다..!! 
#간편하게 하나로 맞춰도 되지만 동시에 여러 개를 돌리고 싶다면 가상 환경을 다르게 설정해도 된다

In [1]:
!pip install openai==1.58.1

  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.5 MB/s  0:00:00
Using cached certifi-2026.6.17-py3-none-any.whl (133 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [openai]14/15 [openai]c]


In [2]:
!pip install openai python-dotenv

---
## 1. LLM API란?

강의자료 요약:

| 구분 | ChatGPT 웹 | API |
|------|------------|-----|
| 사용자 | 사람이 직접 입력 | **프로그램**이 요청 |
| 결과 | 화면에 표시 | **response 객체**로 반환 |
| 본질 | 대화 | **Prompt → Text** |

API 요청의 핵심 3요소:
1. **`model`** — 어떤 LLM을 쓸지
2. **`messages`** — system / user (대화 내용)
3. **`temperature`** — 답변의 무작위성 (0=일관, 1=창의)

---
## 2. API 키 연결


처음에는 **키를 변수에 넣어** 바로 연결해 봅니다.

> OpenAI Platform → [API keys](https://platform.openai.com/api-keys) 에서 발급

In [ ]:
from openai import OpenAI

# 아래에 본인 키를 붙여넣고 실행
api_key = ' '

client = OpenAI(api_key=api_key)

In [9]:
# 연결 테스트
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Hi, reply with one word: OK'}],
    max_tokens=10,
)
print('연결 성공:', test.choices[0].message.content)

연결 성공: OK


1. **`.env` 파일**에 키 저장 (코드와 분리) (OPENAI_API_KEY=sk...)
2. **`.gitignore`**에 `.env` 등록 → Git에 **절대 올리지 않음**

In [50]:
from pathlib import Path

root = Path('..')  # ttest/     ../.. 점점 상위폴더로.
env_path = root / '.env'
#env_path = /Users/user/Desktop/cursor 연습/.env 로 직접 경로 path 추가해도 상관 없음.
gitignore_path = root / '.gitignore'

print('.env 존재:', env_path.exists(), '→', env_path.resolve())
print('.gitignore 존재:', gitignore_path.exists())

if gitignore_path.exists():
    content = gitignore_path.read_text(encoding='utf-8')
    print('.env in .gitignore:', '.env' in content)

.env 존재: True → /Users/user/Desktop/cursor 연습/.env
.gitignore 존재: True
.env in .gitignore: True


In [51]:
from dotenv import load_dotenv
import os

load_dotenv(root / '.env')     #API KEY 호출 ??
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    raise ValueError('ttest/.env에 OPENAI_API_KEY=sk-... 를 설정하세요.')

client = OpenAI(api_key=api_key) #키를 불러옴
print('✅ .env에서 API 키 로드 완료 (코드에 키 없음)')

✅ .env에서 API 키 로드 완료 (코드에 키 없음)


---
## 3. 첫 API 호출 

`chat.completions.create()` — **messages** 리스트를 보내면 AI가 답변합니다.

| role | 역할 |
|------|------|
| `system` | AI 역할·규칙 (선택) |
| `user` | 사용자 질문 |

In [39]:
# 가장 기본적인 기능이라고 함.. 가장 많이 쓸거다?
# role system은 모델한테 명확한 역할을 부여하는 것임
# role user는 직접적으로 물어보는 형태임
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user',   'content': '2002년 월드컵 우승 팀이 어디야?'},
    ],
)

In [40]:
# 웹 ChatGPT에 질문 입력 + 전송  ≈  client.chat.completions.create()
#화면에 나오는 답변            ≈  response.choices[0].message.content

In [41]:
response #모델의 output에 대한 정보

ChatCompletion(id='chatcmpl-DuflUBI2uOckbfWzumk8kj7cYhnPN', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))], created=1782399284, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_de4ea47f89', usage=CompletionUsage(completion_tokens=47, prompt_tokens=30, total_tokens=77, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [42]:
response.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))

In [43]:
response.choices[0].message

ChatCompletionMessage(content='2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[])

In [44]:
response.choices[0].message.content

'2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.'

In [47]:
####### 2022년 월드컵 우승팀 물어보기
response1 = client.chat.completions.create(
    model = 'gpt-4o-mini',
    temperature = 0.1,
    messages = [
        {'role': 'system', 'content' : 'You are a helpful assistant.'},
        {'role': 'user', 'content' : '2022년 월드컵 우승팀은 어디야?'},
    ],
)


...

Ellipsis

In [48]:
response1.choices[0].message.content

'2022년 FIFA 월드컵 우승팀은 아르헨티나입니다. 아르헨티나는 결승에서 프랑스를 상대로 승리하여 36년 만에 월드컵 트로피를 차지했습니다.'

---
## 4. 응답(Response) 객체

API는 **텍스트만**이 아니라 **객체**를 돌려줍니다.

| 필드 | 의미 |
|------|------|
| `response.choices[0].message.content` | **실제 답변 텍스트** ← 가장 많이 사용 |
| `response.choices[0].finish_reason` | `stop`=정상 종료, `length`=토큰 초과 |
| `response.usage.total_tokens` | 이번 호출에 쓴 토큰 수 (비용 참고) |
| `response.id` | 요청 ID (로그·디버깅용) |

In [49]:
print('ID:', response.id)
print('finish_reason:', response.choices[0].finish_reason) #EOS 토큰 도달해서 stop으로 나옴, 토큰 초과로 멈추면 length로 나옴
print('답변:', response.choices[0].message.content)
print('토큰 사용:', response.usage.total_tokens, '(prompt:', response.usage.prompt_tokens,
      '/ completion:', response.usage.completion_tokens, ')')

ID: chatcmpl-DuflUBI2uOckbfWzumk8kj7cYhnPN
finish_reason: stop
답변: 2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.
토큰 사용: 77 (prompt: 30 / completion: 47 )


---
## 5. System / User 프롬프트 

같은 `user` 질문이라도 **`system`에 역할·출력 규칙**을 주면 답변 **형식·깊이·관점**이 달라집니다.

아래는 **의도적으로 짧고 모호한 질문**을 사용합니다. system 유무에 따라 차이가 잘 보입니다.

| | system | user |
|---|--------|------|
| 역할 | 모델의 **역할·규칙·형식** | 이번 **질문·작업** |
| 비유 | 직무 기술서 | 오늘 업무 지시 |

In [26]:
# 질문을 짧고 모호하게 → system이 없으면 범용 답변, 있으면 역할·형식에 맞춘 답변
question = 'for문이 뭐야?'

r1 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[{'role': 'user', 'content': question}],
)

print('=== system 없음 (범용 설명, 길어질 수 있음) ===')
print(r1.choices[0].message.content)

=== system 없음 (범용 설명, 길어질 수 있음) ===
`for`문은 프로그래밍에서 반복문 중 하나로, 특정한 조건을 만족하는 동안 코드를 반복 실행하는 데 사용됩니다. 주로 리스트, 배열, 문자열 등의 요소를 순회하거나, 특정 범위의 숫자를 반복할 때 사용됩니다.

예를 들어, Python에서 `for`문을 사용하는 기본적인 예시는 다음과 같습니다:

```python
# 리스트의 각 요소를 출력하는 예제
fruits = ['apple', 'banana', 'cherry']
for fruit in fruits:
    print(fruit)
```

위의 코드에서는 `fruits` 리스트의 각 요소를 하나씩 `fruit` 변수에 할당하여 출력합니다.

또 다른 예로, 특정 범위의 숫자를 반복하는 경우는 다음과 같습니다:

```python
# 0부터 4까지의 숫자를 출력하는 예제
for i in range(5):
    print(i)
```

이 경우 `range(5)`는 0부터 4까지의 숫자를 생성하며, `for`문은 이 숫자들을 하나씩 `i`에 할당하여 출력합니다.

`for`문은 다양한 프로그래밍 언어에서 비슷한 방식으로 사용되며, 반복적인 작업을 간편하게 처리할 수 있게 해줍니다.


In [27]:
r2 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[
        {
            'role': 'system',
            'content': '''You are a Python tutor.
Rules:
- Answer in Korean only
- Use exactly 3 lines: [비유 1문장] / [코드 예시 max 3 lines] / [실무 한 줄]
- No long explanation, no bullet lists''',
        },
        {'role': 'user', 'content': question},
    ],
)

print('=== system 있음 (튜터 + 3줄 형식 강제) ===')
print(r2.choices[0].message.content)

=== system 있음 (튜터 + 3줄 형식 강제) ===
for문은 반복하는 기계처럼 여러 번 같은 작업을 수행하는 도구입니다.  
```python
for i in range(5):
    print(i)
```
리스트나 배열의 요소를 쉽게 순회할 수 있습니다.


---
## 6. temperature — **일반 예시**

같은 질문, 다른 `temperature` → 결과가 어떻게 달라지는지 확인합니다.

| temperature | 특징 | 용도 |
|-------------|------|------|
| 0 ~ 0.3 | 일관적, 사실 위주 | 분석, 판정, 코드 |
| 0.7 ~ 1.0 | 다양·창의적 | 브레인스토밍, 카피 |

In [29]:
# temperature = 무작위성을 나타내는 항목임, 높을수록 창의적인 답변, 낮을수록 일관적인 답변

question = '팀 워크숍 아이스브레이킹 아이디어 1가지만.'

for temp in [0.0, 0.7, 1.0]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temp,
        messages=[{'role': 'user', 'content': question}],
    )
    print(f'[temperature={temp}] {r.choices[0].message.content}')
    print()

[temperature=0.0] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 두 가지 진실과 하나의 거짓을 말합니다. 다른 팀원들은 어떤 것이 거짓인지 맞춰야 합니다. 이 게임은 서로에 대한 이해를 높이고, 자연스럽게 대화를 유도하는 데 도움이 됩니다. 재미있고 가벼운 분위기를 만들어 팀원 간의 친밀감을 증진시킬 수 있습니다.

[temperature=0.7] "두 진실과 한 거짓" 게임을 제안합니다. 

각 팀원은 자신에 대한 두 가지 진실과 하나의 거짓을 준비합니다. 차례로 자신의 세 가지 사실을 이야기하면, 다른 팀원들은 어떤 것이 거짓인지 맞춰야 합니다. 이 게임은 서로에 대한 흥미로운 사실을 알게 하여 팀원 간의 친밀감을 높이는 데 도움이 됩니다. 또한, 가벼운 경쟁 요소가 있어 분위기를 더욱 즐겁게 만들어줍니다.

[temperature=1.0] "두 진실과 한 거짓" 활동을 추천합니다. 

**방법:**
1. 참가자들은 각자 자신에 대한 두 가지 진실과 하나의 거짓을 준비합니다.
2. 차례대로 각 참가자가 자신의 세 가지 사실을 발표합니다.
3. 다른 팀원들은 어느 것이 거짓인지 맞추는 시간을 가집니다.
4. 정답 발표 후, 각자의 이야기를 나누며 재미있고 친밀한 분위기를 형성할 수 있습니다.

이 활동은 팀원 간의 이해도를 높이고, 서로에 대한 호기심을 자극하는 데 효과적입니다!



---
## 7. `max_tokens` — 출력 길이 제한

답변이 너무 길어지는 것을 막거나, 비용을 줄일 때 사용합니다.

In [30]:
for max_tok in [20, 100]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0.3,
        max_tokens=max_tok,
        messages=[{'role': 'user', 'content': '대한민국의 역사를 요약해줘.'}],
    )
    print(f'--- max_tokens={max_tok} (finish: {r.choices[0].finish_reason}) ---')
    print(r.choices[0].message.content)
    print()

--- max_tokens=20 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 포함

--- max_tokens=100 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 포함하고 있습니다. 아래는 대한민국의 주요 역사적 사건을 요약한 것입니다.

1. **고대 및 삼국시대 (기원전 2333년 ~ 668년)**:
   - 고조선: 전설적인 단군이 세운 고조선이 기원전 2333년에 건국되었다고 전해진다.
   - 삼국



---
## 8. `ask()` 함수 — 재사용 패턴

같은 API 호출을 함수로 감싸기.

In [ ]:
def ask(user_message, system_message='You are a helpful assistant.', temperature=0.2, max_tokens=None):
    """1턴 질의 — 답변 텍스트만 반환"""
    kwargs = dict(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_message},
            {'role': 'user',   'content': user_message},
        ],
    )
    if max_tokens is not None:
        kwargs['max_tokens'] = max_tokens
    response = client.chat.completions.create(**kwargs) #** 있으면 딕셔너리를 풀어서 각각 전달
    return response.choices[0].message.content


print(ask('서울 3일 여행 코스 추천해줘. bullet 3개만.', max_tokens=150))

서울 3일 여행 코스 추천:

- **1일차: 역사와 문화 탐방**
  - 경복궁 방문: 고궁 탐방과 국립민속박물관 관람
  - 북촌 한옥마을 산책: 전통 한옥과 예쁜 카페 탐방
  - 인사동 거리: 전통 공예품 쇼핑 및 찻집 체험

- **2일차: 현대 서울의 매력**
  - 명동 쇼핑: 다양한 브랜드와 길거리 음식 즐기기
  - 남산서울타워: 서울 전경 감상 및 케이블카 체험
  - 홍대 거리: 젊은 문화와 예술


In [32]:
print(ask('안녕, 내이름은 탬탬이야', max_tokens=150))

안녕하세요, 탬탬님! 만나서 반가워요. 어떻게 도와드릴까요?


In [33]:
print(ask('안녕, 내이름은 뭐라구?', max_tokens=150)) #앞의 대화가 뒤의 대화에 이어지지 않는다.

안녕하세요! 당신의 이름은 제가 알 수 없어요. 당신의 이름을 알려주실 수 있나요?


---
## 9. 멀티턴 대화 

이전 대화를 `messages`에 **그대로 이어 붙이면** 후속 질문이 가능합니다.

```
system → user → assistant → user → ...
```

### 기존대화

In [34]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '2일차만 더 자세히. 맛집 2곳 포함.'},
]

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
**1일차:** 제주공항 도착 후, 한라산 국립공원에서 하이킹을 즐기고, 저녁에는 제주시의 흑돼지 구이를 맛본다.  

**2일차:** 성산일출봉에서 일출을 감상한 후, 우도 섬으로 배를 타고 가서 자전거를 타며 섬을 탐방한다.  

**3일차:** 만장굴과 주상절리대를 방문한 후, 제주 오일장(재래시장)에서 쇼핑과 먹거리를 즐기고, 공항으로 돌아간다.  

=== 2턴 (맥락 유지) ===
물론입니다! 2일차 일정을 자세히 안내해 드리겠습니다.

### 2일차 일정

**아침:**
- **조식**: 호텔에서 제공하는 조식을 즐기세요. 다양한 메뉴가 있을 것입니다.

**오전:**
- **관광지 1: 경복궁**
  - 경복궁은 조선 왕조의 주요 궁궐로, 아름다운 건축물과 정원이 있습니다. 궁궐 내에서 가이드 투어를 이용하면 역사에 대한 더 깊은 이해를 할 수 있습니다.
  - **추천 활동**: 수문장 교대식 관람

**점심:**
- **맛집 1: 광화문 국밥**
  - 위치: 경복궁 근처
  - 메뉴: 뜨끈한 국밥과 함께 다양한 반찬이 제공됩니다. 특히, 소고기 국밥이 인기입니다.

**오후:**
- **관광지 2: 북촌 한옥마을**
  - 전통 한옥이 잘 보존된 지역으로, 한옥을 구경하며 산책하기 좋습니다. 사진 찍기에도 좋은 장소입니다.
  - **추천 활동**: 한옥 체험 프로그램 참여

**저녁:**
- **맛집 2: 삼청동 수제비**
  - 위치: 삼청동
  - 메뉴: 신선한 재료로 만든 수제비와 함께 다양한 찌개를 즐길 수 있습니다. 분위기도 아늑하여 저녁 식사에 적합합니다.

**저녁 후:**
- **산책**: 삼청동 거리에서 여유롭게 산책하며 카페에 들러 디저트를 즐기세요. 

이렇게 2일차 일정을 구성해 보았습니다. 추가적인 질문이 있으시면 언제든지 말씀해 주세요!


### 멀티턴

### Assistant 메시지는 앞서 있었던 prompt들(user/system)에 대한 모델의 응답을 구성하며, 종종 대화 상태를 유지하기 위해 히스토리의 일부로 저장됩니다.

#### 특징
모델이 생성한 응답을 포함한다.

이전 message의 context 처리를 위해 대화의 연속성 유지를 돕는다.

다음 API 호출에서는 assistant 메시지를 포함시켜야 문맥이 이어짐.

In [35]:
# 그 전 내용들이 input으로 들어가야 맥락이 유지된다.
# 한 단어 한 단어씩 만들고 다시 input으로 들어가서 생성을 한다.
# 첫 번째 질문에 대한 답도 input으로 다시 append 해야만 맥락이 이어진다.

messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages.append({'role': 'assistant', 'content': answer1})
messages.append({'role': 'user', 'content': '2일차만 더 자세히. 맛집 2곳 포함.'})

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
제주도 2박 3일 여행 계획입니다.

**1일차:** 제주공항 도착 후 한라산 등반 및 성산일출봉 관람, 저녁에는 제주 전통 음식인 흑돼지 BBQ 즐기기.

**2일차:** 협재해수욕장에서 해수욕 및 해양 스포츠 체험 후, 오설록 티 뮤지엄 방문하여 녹차 아이스크림 맛보기, 저녁에는 중문관광단지 탐방.

**3일차:** 만장굴 탐방 후, 제주 민속촌에서 제주 전통 문화 체험, 마지막으로 공항으로 이동하여 여행 마무리.

=== 2턴 (맥락 유지) ===
2일차 자세한 일정입니다.

**오전:** 협재해수욕장으로 이동하여 해수욕과 해양 스포츠를 즐깁니다. 맑은 바다에서 수영이나 스노클링을 해보세요.

**점심:** 협재해수욕장 근처의 맛집 '협재해물탕'에서 신선한 해물탕을 맛보세요. 해물의 풍미가 가득한 요리로 유명합니다.

**오후:** 오설록 티 뮤지엄으로 이동하여 제주 녹차의 역사와 문화를 배우고, 다양한 녹차 제품을 시식해보세요. 특히 녹차 아이스크림이 인기입니다.

**저녁:** 중문관광단지 내의 '중문횟집'에서 신선한 회와 해산물 요리를 즐깁니다. 제주에서 잡은 신선한 재료로 만든 요리를 맛볼 수 있습니다.

**저녁 후:** 중문관광단지 주변을 산책하며 아름다운 야경을 감상하고, 카페에서 제주 특산물인 한라봉 음료를 즐기며 하루를 마무리합니다.


---
## ✏️ 실습 1

아래 주제로 **2턴** 대화를 만들어 보세요.

1턴: "Python for문 기본 문법 설명해줘"
2턴: "방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘"

In [53]:
messages = [
    {'role': 'system', 'content': 'You are a Python tutor. Answer in Korean.'},
    {'role': 'user', 'content': 'Python for문 기본 문법 설명해줘'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
answer1


"Python의 `for` 문은 반복(iteration)을 수행하는 데 사용됩니다. 주로 리스트, 튜플, 문자열과 같은 iterable 객체를 순회할 때 사용됩니다. 기본 문법은 다음과 같습니다.\n\n```python\nfor 변수 in iterable:\n    # 실행할 코드\n```\n\n여기서 `변수`는 iterable의 각 요소를 참조하는 변수이며, `iterable`은 반복할 수 있는 객체입니다. `for` 문 안의 코드는 `iterable`의 각 요소에 대해 한 번씩 실행됩니다.\n\n예를 들어, 리스트의 각 요소를 출력하는 간단한 예시는 다음과 같습니다.\n\n```python\nfruits = ['사과', '바나나', '체리']\n\nfor fruit in fruits:\n    print(fruit)\n```\n\n위 코드를 실행하면 다음과 같은 결과가 출력됩니다.\n\n```\n사과\n바나나\n체리\n```\n\n또한, `range()` 함수를 사용하여 특정 범위의 숫자를 반복할 수도 있습니다. 예를 들어, 0부터 4까지의 숫자를 출력하는 코드는 다음과 같습니다.\n\n```python\nfor i in range(5):\n    print(i)\n```\n\n이 코드를 실행하면 다음과 같은 결과가 출력됩니다.\n\n```\n0\n1\n2\n3\n4\n```\n\n이처럼 `for` 문은 다양한 iterable 객체를 순회하며 반복적인 작업을 수행하는 데 유용하게 사용됩니다."

In [55]:
messages.append({'role': 'assistant', 'content': 'answer1'})
messages.append({'role': 'user', 'content': '방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘'})
messages

[{'role': 'system', 'content': 'You are a Python tutor. Answer in Korean.'},
 {'role': 'user', 'content': 'Python for문 기본 문법 설명해줘'},
 {'role': 'assistant', 'content': 'answer1'},
 {'role': 'user', 'content': '방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘'}]

In [58]:
r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer2 = r2.choices[0].message.content


In [59]:
print(answer1)
print(answer2)

Python의 `for` 문은 반복(iteration)을 수행하는 데 사용됩니다. 주로 리스트, 튜플, 문자열과 같은 iterable 객체를 순회할 때 사용됩니다. 기본 문법은 다음과 같습니다.

```python
for 변수 in iterable:
    # 실행할 코드
```

여기서 `변수`는 iterable의 각 요소를 참조하는 변수이며, `iterable`은 반복할 수 있는 객체입니다. `for` 문 안의 코드는 `iterable`의 각 요소에 대해 한 번씩 실행됩니다.

예를 들어, 리스트의 각 요소를 출력하는 간단한 예시는 다음과 같습니다.

```python
fruits = ['사과', '바나나', '체리']

for fruit in fruits:
    print(fruit)
```

위 코드를 실행하면 다음과 같은 결과가 출력됩니다.

```
사과
바나나
체리
```

또한, `range()` 함수를 사용하여 특정 범위의 숫자를 반복할 수도 있습니다. 예를 들어, 0부터 4까지의 숫자를 출력하는 코드는 다음과 같습니다.

```python
for i in range(5):
    print(i)
```

이 코드를 실행하면 다음과 같은 결과가 출력됩니다.

```
0
1
2
3
4
```

이처럼 `for` 문은 다양한 iterable 객체를 순회하며 반복적인 작업을 수행하는 데 유용하게 사용됩니다.
아래는 1부터 5까지의 합을 구하는 `for`문을 사용한 코드 예시입니다.

```python
합 = 0
for i in range(1, 6):  # 1부터 5까지의 숫자를 반복
    합 += i  # 현재 숫자를 합에 더함

print(합)  # 결과 출력
```

이 코드를 실행하면 1부터 5까지의 합인 15가 출력됩니다.


In [ ]:
# ── 여기에 작성 ──
messages = [
    {'role': 'system', 'content': 'You are a Python tutor. Answer in Korean.'},
    # {'role': 'user', 'content': '...'},
]
# r1 = client.chat.completions.create(...)
# messages.append(...)
# r2 = client.chat.completions.create(...)

---
## ✏️ 실습 1

커서 프롬프트 : 3일차 폴더에 스트림릿과 실습코드의 openai api를 이용해서 간단한 챗봇을 만드는 python 코드 만들어줘

In [36]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 14.7 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 6.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 8.1 MB/s  0:00:00m0:00:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 12.2 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 21.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 19.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.2/31.2 MB 3.0 MB/s  0:00:10m0:00:0100:01m
  Attempting uninstall: packagingm━━━━━━━━━━━━━━━━━━━━━━━━━━━  9/30 [pillow]]
    Found existing installation: packaging 26.2━━━━━━━━━━━━━━━  9/30 [pillow]
    Uninstalling packaging-26.2:90m━━━━━━━━━━━━━━━━━━━━━━━━━━━  9/30 [pillow]
      Successfully uninstalled packaging-26.2━━━━━━━━━━━━━━━━━  9/30 [pillow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30/30 [streamlit]30 [streamlit]
